# Path A: Load-Firing + CEBRA Cross-Species Manifold Comparison

## Scope and framing

Watters et al. 2026 (*Working Memory of Multi-Object Scenes in Primate Frontal Cortex*) established that macaque DMFC supports **gain coding** for multi-object working memory, with weighted-sum representations outperforming slot and switching models on cross-validated neural data and predicting behavioral errors. They explicitly leave open whether human frontal cortex implements a similar regime.

This notebook does **not** test slot vs. gain in humans. With n=17 frontal concept cells, we don't have power for that, and our prior analyses showed the conclusion would be unsupportable.

Instead, this is a **phenomenological cross-species comparison**:

1. **Load-on-firing analysis**: Does delay-period firing rate change with memory load in both species, in matched directions, after appropriate controls?
2. **CEBRA manifold alignment**: Conditioned on shared task labels (load, item-present-or-absent, trial outcome), do the population-level manifolds during maintenance share recognizable structure across species?

Together these support a claim of the form: *frontal cortex maintenance representations show convergent rate-level and manifold-level signatures across species despite task structural differences (sequential Sternberg vs. simultaneous spatial array) and recording region differences (pre-SMA/dACC vs. DMFC).*

## Important caveats baked into this analysis

- **Task mismatch.** Human task is sequential 3-image Sternberg (serial-order encoding); macaque task is simultaneous 3-object spatial array. The shared structural axis is *load* (1, 2, or 3 items maintained); item-binding differs (time vs. space). Cross-species claims must be at the load and item-present-or-absent level, not the position/binding level.
- **Region mismatch.** Human cells are pre-SMA/dACC frontal concept cells. Macaque cells are DMFC. These are nearby but not homologous regions — pre-SMA/dACC in humans is closer to medial frontal cortex; DMFC in macaque is a broader medial-frontal designation that includes pre-SMA-equivalent areas. Worth flagging in writeup.
- **Pre-SMA concept cells are unusual.** Kaminski et al. 2017 explicitly reported *no* concept cells in pre-SMA in their cohort. The 2024 Kyzar release uses different selection criteria. Verify how the `*_concept_cells.csv` files define these before drawing strong conclusions.
- **Selection bias.** Macaque cells are filtered to identity-tuned (Identity AUC > 0.65, p < 0.05). Human cells are pre-selected concept cells (image-selective). Both selections bias toward cells that respond to specific items, which is appropriate for the question but means we're comparing *item-selective subpopulations*, not the regions as a whole.
- **Trial-level vs. cell-level variance.** Bootstrap CIs in this notebook are over cell resampling. They do not reflect within-cell trial variance, which is already averaged into per-cell condition means. This understates true uncertainty; treat CIs as lower bounds on real variability.
- **CEBRA alignment is suggestive, not mechanistic.** Manifold alignment under shared labels is consistent with shared computation but does not prove it. Convergent task demands (item maintenance, load handling, response preparation) can produce shared manifolds without shared underlying mechanisms.

## Cell 1 — Imports and setup

In [ ]:
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from pynwb import NWBHDF5IO
from tqdm import tqdm

# CEBRA — install if not present: pip install cebra
import cebra
from cebra import CEBRA

warnings.filterwarnings('ignore')

# Reproducibility
RNG_SEED = 42
np.random.seed(RNG_SEED)

# Macaque dataset path
sys.path.append('../multi_object_memory_2025')
sys.path.append('../multi_object_memory_2025/phys_modeling')
sys.path.append('../multi_object_memory_2025/phys_modeling/training')
try:
    from phys_modeling.training.dataset import Dataset as WattersDataset
except ImportError:
    print('[WARN] Could not import Watters Dataset. Macaque pipeline will be skipped.')
    WattersDataset = None

# Path config — adjust to your local paths
HUMAN_DATA_ROOT  = '/Volumes/fetty/000469'
HUMAN_CC_DIR     = '/Users/cwook/Documents/humac/data/concept_cells'
MAC_INVENTORY    = '/Users/cwook/Documents/humac/lists/task_shape_inventory_survivors.csv'
OUTPUT_DIR       = '/Users/cwook/Documents/humac/output/path_a'
os.makedirs(OUTPUT_DIR, exist_ok=True)

HUMAN_REGIONS = [
    'pre_supplementary_motor_area_right',
    'pre_supplementary_motor_area_left',
    'dorsal_anterior_cingulate_cortex_right',
    'dorsal_anterior_cingulate_cortex_left',
]

# Sliding-window params for human extraction
WINDOW_SIZE = 0.5    # seconds
STEP_SIZE   = 0.05   # seconds

print('Setup complete.')

## Cell 2 — Data extraction (human Sternberg + macaque DMFC)

Same extraction logic as the previous notebooks. Each cell becomes a tuple of (X_trials, y_binned, loads, [pref_id]). Trial-level neural data is preserved (not collapsed across the delay) so the load and CEBRA analyses can both work on the same source.

In [ ]:
def get_sliding_window_counts(spike_times, t0, t1, window=WINDOW_SIZE, step=STEP_SIZE):
    spike_times = np.asarray(spike_times)
    bin_starts = np.arange(t0, t1 - window, step)
    counts = np.zeros(len(bin_starts))
    centers = bin_starts + window / 2.0
    for i, ts in enumerate(bin_starts):
        counts[i] = np.sum((spike_times >= ts) & (spike_times < ts + window))
    return counts, centers


def extract_human_cells(data_root, cc_dir, regions, window=WINDOW_SIZE, step=STEP_SIZE):
    cells = {}
    master_t = None
    csv_files = glob.glob(os.path.join(cc_dir, '*_concept_cells.csv'))
    print(f'[HUMAN] Scanning {len(csv_files)} concept-cell CSVs...')

    for csv_path in tqdm(csv_files, desc='Human CSVs'):
        df_cc = pd.read_csv(csv_path)
        df_target = df_cc[df_cc['location'].isin(regions)]
        if df_target.empty:
            continue
        sub, ses = os.path.basename(csv_path).split('_')[:2]
        nwb_files = glob.glob(os.path.join(data_root, '**', f'{sub}_{ses}*.nwb'), recursive=True)
        if not nwb_files:
            continue
        with NWBHDF5IO(nwb_files[0], 'r') as io:
            nwb = io.read()
            trials = nwb.trials.to_dataframe().dropna(
                subset=['timestamps_Maintenance', 'timestamps_Probe']
            )
            for _, cc in df_target.iterrows():
                if pd.isna(cc['preferred_image_id']):
                    continue
                uid, loc, pref = int(cc['unit_id']), cc['location'], int(cc['preferred_image_id'])
                try:
                    spikes = nwb.units.get_unit_spike_times(uid)
                except Exception:
                    continue
                Xs, Ys, Ls, Cs = [], [], [], []
                for _, tr in trials.iterrows():
                    t0, t1 = tr['timestamps_Maintenance'], tr['timestamps_Probe']
                    L = int(tr['loads'])
                    counts, centers = get_sliding_window_counts(spikes, t0, t1, window, step)
                    if master_t is None:
                        master_t = centers - t0
                    target_len = len(master_t)
                    if len(counts) > target_len:
                        counts = counts[:target_len]
                    elif len(counts) < target_len:
                        counts = np.pad(counts, (0, target_len - len(counts)), 'constant',
                                        constant_values=np.nan)
                    seq = [0, 0, 0]
                    if L >= 1: seq[0] = int(tr['loadsEnc1_PicIDs'])
                    if L >= 2: seq[1] = int(tr['loadsEnc2_PicIDs'])
                    if L >= 3: seq[2] = int(tr['loadsEnc3_PicIDs'])
                    Xs.append(seq)
                    Ys.append(counts)
                    Ls.append(L)
                    # Outcome: response_correct or similar — Kyzar dataset uses 'response_correct'
                    correct = int(tr['response_correct']) if 'response_correct' in tr else -1
                    Cs.append(correct)
                key = f'{sub}_{ses}_unit{uid}_{loc}'
                cells[key] = (np.array(Xs), np.array(Ys), np.array(Ls), pref, np.array(Cs))
    print(f'[HUMAN] Extracted {len(cells)} cells.')
    return cells, master_t


def extract_macaque_cells(session_list):
    if WattersDataset is None:
        return {}, None
    cells = {}
    cmap = {'a': 1, 'b': 2, 'c': 3}
    master_t = None
    id_filter = lambda df: df[
        (df['quality'].isin(['good', '1', 'sua']))
        & (df['Identity AUC'] > 0.65)
        & (df['Identity p-value'] < 0.05)
        & (df['probe'] == 's0')
    ]
    print(f'[MAC] Processing {len(session_list)} sessions...')
    for sub, ses in tqdm(session_list, desc='Macaque sessions'):
        try:
            ds = WattersDataset(subject=sub, session=ses, phase='delay',
                                 unit_filter=id_filter, trial_filter=lambda df: df,
                                 max_n_objects=3)
            neural = ds.data['neural']  # (n_trials, n_bins, n_neurons)
            if neural.shape[2] == 0:
                continue
            trials_df = None
            for v in (ds.data.values() if isinstance(ds.data, dict) else []):
                if isinstance(v, pd.DataFrame):
                    trials_df = v; break
            if trials_df is None:
                for a in dir(ds):
                    v = getattr(ds, a, None)
                    if isinstance(v, pd.DataFrame):
                        trials_df = v; break
            if trials_df is None:
                continue
            if len(trials_df) != neural.shape[0] and 'completed' in trials_df.columns:
                trials_df = trials_df[trials_df['completed'] == True]
            n_bins = neural.shape[1]
            if master_t is None:
                master_t = ds.data['time'] if ('time' in ds.data and len(ds.data['time']) == n_bins) \
                    else np.linspace(0, 1.0, n_bins)
            tlen = len(master_t)
            if neural.shape[1] > tlen:
                neural = neural[:, :tlen, :]
            elif neural.shape[1] < tlen:
                neural = np.pad(neural, ((0, 0), (0, tlen - neural.shape[1]), (0, 0)),
                                'constant', constant_values=np.nan)
            X_tr, L_tr, C_tr = [], [], []
            for _, row in trials_df.iterrows():
                arr = [0, 0, 0]
                L = int(row['num_objects'])
                for i in range(L):
                    loc_col, id_col = f'object_{i}_location', f'object_{i}_id'
                    if pd.notna(row.get(loc_col)) and pd.notna(row.get(id_col)):
                        arr[int(row[loc_col])] = cmap[row[id_col]]
                X_tr.append(arr); L_tr.append(L)
                # Outcome — Watters dataset has 'correct' or similar; fallback -1
                if 'correct' in row:
                    C_tr.append(int(row['correct']))
                elif 'response_correct' in row:
                    C_tr.append(int(row['response_correct']))
                else:
                    C_tr.append(-1)
            X_tr, L_tr, C_tr = np.array(X_tr), np.array(L_tr), np.array(C_tr)
            for n in range(neural.shape[2]):
                cells[f'{sub}_{ses}_unit{n}'] = (X_tr, neural[:, :, n], L_tr, None, C_tr)
        except Exception as e:
            continue
    print(f'[MAC] Extracted {len(cells)} cells.')
    return cells, master_t

# Run extractions
human_cells, human_t = extract_human_cells(HUMAN_DATA_ROOT, HUMAN_CC_DIR, HUMAN_REGIONS)

try:
    inv = pd.read_csv(MAC_INVENTORY)
    mac_sessions = [
        (r['file'].split('/')[0].replace('sub-', ''),
         r['file'].split('/')[1].split('_')[1].replace('ses-', ''))
        for _, r in inv[inv['shape'] == 'Triangle'].iterrows()
    ]
    mac_cells, mac_t = extract_macaque_cells(mac_sessions)
except FileNotFoundError:
    print(f'[WARN] Inventory not found at {MAC_INVENTORY}.')
    mac_cells, mac_t = {}, None

print(f'\nFinal counts: human={len(human_cells)} cells, macaque={len(mac_cells)} cells')
print(f'Time bins: human={len(human_t) if human_t is not None else 0}, '
      f'macaque={len(mac_t) if mac_t is not None else 0}')

## Cell 3 — Analysis 1: Load-on-firing-rate, with controls

**Question:** Does delay-period firing rate change with load in both species, and in matched directions?

**Method:**
- Per-cell mean firing rate during the delay period, conditioned on load and on whether the preferred item was present (human) or whether any item was present at the cell's preferred position (macaque).
- Use **z-scored within-cell** firing rates so cells with different baseline rates contribute equally.
- Statistics: **linear mixed-effects model** with load as fixed effect and cell as random effect (clusters cells properly), separately per species. Report β_load and 95% CI.
- Bootstrap CIs over cell resampling for the per-load means in figures.

**Why this controls for the global firing-shift confound (partially):**
- Conditioning on preferred-item-present vs. absent gives us a *within-load* contrast that isolates item-selective firing from global rate shifts.
- If load affects global rate but not the present-vs-absent contrast, that's a generic effect, not a maintenance signature.
- If load specifically attenuates the present-vs-absent contrast, that's the gain-like signature (without claiming gain mechanism).

In [ ]:
from statsmodels.regression.mixed_linear_model import MixedLM

def per_trial_delay_rate(y_binned):
    """Average across delay-period bins -> single rate per trial."""
    return np.nanmean(y_binned, axis=1)


def build_load_table(cells, species):
    """Long-format dataframe: one row per (cell, trial). Columns: cell, load, present, correct, rate_z."""
    rows = []
    for key, data in cells.items():
        X, y, loads, pref, correct = data
        rates = per_trial_delay_rate(y)
        # z-score within cell across all trials
        if np.nanstd(rates) > 0:
            rates_z = (rates - np.nanmean(rates)) / np.nanstd(rates)
        else:
            rates_z = rates - np.nanmean(rates)
        if species == 'human':
            present = np.any(X == pref, axis=1).astype(int)
        else:
            # Macaque: 'present' = any nonzero (always true for L>=1); use a different contrast.
            # Better: average across positions for the cell's preferred item.
            # Without identity-tuning per-cell info, use load only here.
            present = np.ones(len(loads), dtype=int)
        for i in range(len(loads)):
            if np.isnan(rates_z[i]):
                continue
            rows.append({
                'cell': key,
                'load': int(loads[i]),
                'present': int(present[i]),
                'correct': int(correct[i]) if i < len(correct) else -1,
                'rate_z': float(rates_z[i]),
                'rate_raw': float(rates[i]),
                'species': species,
            })
    return pd.DataFrame(rows)

df_human = build_load_table(human_cells, 'human')
df_mac   = build_load_table(mac_cells, 'macaque')
df_all   = pd.concat([df_human, df_mac], ignore_index=True)
print(f'Trials: human={len(df_human)}, macaque={len(df_mac)}')
print(df_all.groupby(['species', 'load']).size().unstack(fill_value=0))

In [ ]:
# Mixed-effects fits, per species
def fit_load_lme(df, label):
    """rate_z ~ load + (1 | cell). Restrict to correct trials when available."""
    sub = df[df['correct'].isin([1, -1])].copy()
    if len(sub) == 0:
        print(f'[{label}] No data after filter.')
        return None
    sub['load_c'] = sub['load'] - 1  # center so intercept = load-1 mean
    md = MixedLM.from_formula('rate_z ~ load_c', groups='cell', data=sub)
    res = md.fit(method='lbfgs')
    print(f'\n--- {label} (n_trials={len(sub)}, n_cells={sub["cell"].nunique()}) ---')
    print(res.summary().tables[1])
    return res

res_hum = fit_load_lme(df_human, 'HUMAN load->rate (z)')
res_mac = fit_load_lme(df_mac,   'MACAQUE load->rate (z)')

# Present-vs-absent x load interaction (human only — macaque doesn't have natural absent trials)
if len(df_human) > 0:
    sub = df_human[df_human['correct'].isin([1, -1])].copy()
    sub['load_c'] = sub['load'] - 1
    md = MixedLM.from_formula('rate_z ~ load_c * present', groups='cell', data=sub)
    res_hum_int = md.fit(method='lbfgs')
    print('\n--- HUMAN load x present interaction ---')
    print(res_hum_int.summary().tables[1])

In [ ]:
# Plot: per-load mean firing rate (z-scored), with bootstrap CIs over cell resampling
def boot_per_load(df, n_boot=1000):
    cells = df['cell'].unique()
    out = {1: [], 2: [], 3: []}
    for _ in range(n_boot):
        boot_cells = np.random.choice(cells, len(cells), replace=True)
        sub = df[df['cell'].isin(boot_cells)]
        # weighted mean per load, weighted by within-cell trial count
        for L in [1, 2, 3]:
            v = sub[sub['load'] == L]['rate_z']
            out[L].append(v.mean() if len(v) > 0 else np.nan)
    return out

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, df_sp, lbl in zip(axes, [df_human, df_mac], ['Human pre-SMA/dACC', 'Macaque DMFC']):
    if len(df_sp) == 0:
        ax.set_title(f'{lbl}: no data'); continue
    boots = boot_per_load(df_sp)
    means = [np.nanmean(boots[L]) for L in [1, 2, 3]]
    ci_lo = [np.nanpercentile(boots[L], 2.5) for L in [1, 2, 3]]
    ci_hi = [np.nanpercentile(boots[L], 97.5) for L in [1, 2, 3]]
    ax.errorbar([1, 2, 3], means,
                yerr=[np.array(means) - np.array(ci_lo), np.array(ci_hi) - np.array(means)],
                fmt='o-', lw=2, capsize=5, color='#1f77b4')
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_xlabel('Load')
    ax.set_ylabel('Delay-period firing (z-scored within cell)')
    ax.set_title(f'{lbl}\n(n={df_sp["cell"].nunique()} cells)')
    ax.set_xticks([1, 2, 3])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A1_load_firing.pdf'))
plt.show()

## Cell 4 — Analysis 2: CEBRA cross-species manifold alignment

**Question:** Conditioned on shared task labels (load, trial outcome), does the population-level manifold during maintenance share recognizable structure across species?

**Method:**
1. Build per-species **trial-by-time-bin-by-cell** tensors. Treat each (trial, time-bin) as a sample.
2. Train CEBRA-Time (continuous time + behavior labels) **separately per species** with shared label structure (load, outcome). This gives each species its own latent embedding conditioned on the same labels.
3. Compare embeddings:
   - **Within-species structure**: silhouette score of load clusters in each latent space. Are loads separable in each species' manifold?
   - **Cross-species alignment**: Procrustes / linear-map alignment between the two label-conditioned centroids (one per (load, outcome) combination). Quantify residual alignment error after orthogonal rotation.
   - **Cross-species decoding**: Train a load decoder on macaque embeddings, test on human embeddings (after Procrustes alignment). If load is decodable across species, the manifolds share usable structure.
4. Report a permutation null for the cross-species decoder (shuffle human labels, recompute alignment + decoding accuracy).

**What this tests:** whether the manifold's organization with respect to load is recognizably similar across species. Not whether the underlying mechanism is the same — convergent task demands can produce convergent manifolds.

**What can break this:**
- Different cell counts (17 vs ~128) → low-dim CEBRA may not be comparable. We use the same `output_dimension` for both, but the human side will be noisier.
- Different time-bin durations and counts. We resample both to a common time grid before training.
- Trials are not balanced across loads (load-1 typically more common). We balance by stratified subsampling before training.

In [ ]:
def build_cebra_tensor(cells, species, common_time_bins=None):
    """
    Construct trial-by-bin-by-cell pseudopopulation tensor with aligned labels.
    Returns:
        neural   : (n_samples, n_cells)  — flat across (trial, time)
        labels   : DataFrame with columns trial_idx, time_idx, load, present, correct, time_within_delay
    Trials with NaN entries in any cell are dropped (pseudopopulation requirement).
    """
    cell_keys = list(cells.keys())
    if not cell_keys:
        return None, None
    # Use the trial structure of the first cell. CRITICAL ASSUMPTION:
    # In macaque each session has identical trials_df across cells (same session); pseudopopulating
    # across sessions discards real trial alignment. We build trial pseudo-IDs, not real trial IDs.
    # For human Sternberg, each cell is its own session, so trials are independent per cell.
    n_bins = cells[cell_keys[0]][1].shape[1]
    # Align all to min trial count? Better: use trial-averaged-by-condition pseudopopulation.
    # We construct condition-binned data: for each (load, present, correct) combo, average
    # the per-cell PSTH across trials of that combo, then concatenate cells.
    # Then expose (condition, time) as samples.
    if species == 'human':
        conds = [(L, p) for L in [1, 2, 3] for p in [0, 1]]
        cond_names = [f'L{L}_P{p}' for L, p in conds]
    else:
        # Macaque: condition by load only (no present/absent contrast available the same way)
        conds = [(L, 1) for L in [1, 2, 3]]
        cond_names = [f'L{L}' for L, _ in conds]

    n_cond = len(conds)
    n_cells = len(cell_keys)
    psth = np.full((n_cond, n_bins, n_cells), np.nan)
    for ci, (L_target, p_target) in enumerate(conds):
        for ni, key in enumerate(cell_keys):
            X, y, loads, pref, correct = cells[key]
            if species == 'human':
                present = np.any(X == pref, axis=1).astype(int)
                mask = (loads == L_target) & (present == p_target)
            else:
                mask = (loads == L_target)
            # Restrict to correct trials when available
            if np.any(correct >= 0):
                mask = mask & (correct == 1)
            if np.sum(mask) > 0:
                psth[ci, :, ni] = np.nanmean(y[mask, :], axis=0)
    # Impute missing cells per condition with cell mean (avoid biasing with zero)
    for ni in range(n_cells):
        flat = psth[:, :, ni]
        if np.isnan(flat).any():
            cell_mean = np.nanmean(flat)
            psth[np.isnan(psth[:, :, ni]).nonzero()[0],
                 np.isnan(psth[:, :, ni]).nonzero()[1], ni] = cell_mean if not np.isnan(cell_mean) else 0
    # Reshape to (n_cond * n_bins, n_cells)
    neural = psth.transpose(0, 1, 2).reshape(n_cond * n_bins, n_cells)
    labels = pd.DataFrame({
        'condition': np.repeat(cond_names, n_bins),
        'load': np.repeat([L for L, _ in conds], n_bins),
        'present': np.repeat([p for _, p in conds], n_bins),
        'time_idx': np.tile(np.arange(n_bins), n_cond),
    })
    return neural, labels

human_neural, human_labels = build_cebra_tensor(human_cells, 'human')
mac_neural, mac_labels     = build_cebra_tensor(mac_cells, 'macaque')

if human_neural is not None:
    print(f'Human CEBRA input: {human_neural.shape} samples x cells; {human_labels["condition"].nunique()} conditions')
if mac_neural is not None:
    print(f'Macaque CEBRA input: {mac_neural.shape} samples x cells; {mac_labels["condition"].nunique()} conditions')

In [ ]:
# Train CEBRA — supervised on load, with time as auxiliary continuous label
OUT_DIM = 4    # latent dimensionality. Keep modest given low n_cells on human side.
MAX_ITER = 3000

def train_cebra(neural, labels, output_dim=OUT_DIM, max_iter=MAX_ITER):
    if neural is None:
        return None, None
    model = CEBRA(
        model_architecture='offset10-model',
        batch_size=min(64, max(8, neural.shape[0] // 8)),
        learning_rate=3e-4,
        temperature=1.0,
        output_dimension=output_dim,
        max_iterations=max_iter,
        distance='cosine',
        conditional='time_delta',
        device='cpu',
        verbose=True,
        time_offsets=10,
    )
    # Discrete label: load. Continuous label: time within condition.
    discrete = labels['load'].to_numpy().astype(int)
    continuous = labels['time_idx'].to_numpy().astype(float).reshape(-1, 1)
    model.fit(neural, continuous, discrete)
    embedding = model.transform(neural)
    return model, embedding

print('Training CEBRA — human...')
model_h, emb_h = train_cebra(human_neural, human_labels)
print('\nTraining CEBRA — macaque...')
model_m, emb_m = train_cebra(mac_neural, mac_labels)

if emb_h is not None: print(f'Human embedding: {emb_h.shape}')
if emb_m is not None: print(f'Macaque embedding: {emb_m.shape}')

In [ ]:
# Visualize embeddings — first 2-3 latent dimensions, colored by load
def plot_embedding(emb, labels, ax, title):
    if emb is None:
        ax.set_title(f'{title}: no data'); return
    colors = {1: '#2ca02c', 2: '#ff7f0e', 3: '#d62728'}
    # PCA to 2D for plotting
    pca = PCA(n_components=2).fit_transform(emb)
    for L in [1, 2, 3]:
        m = labels['load'].to_numpy() == L
        ax.scatter(pca[m, 0], pca[m, 1], s=10, alpha=0.5, color=colors[L], label=f'Load {L}')
    ax.set_xlabel('CEBRA dim 1 (PCA)')
    ax.set_ylabel('CEBRA dim 2 (PCA)')
    ax.set_title(title)
    ax.legend(loc='best', fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_embedding(emb_h, human_labels, axes[0], f'Human (n={len(human_cells)} cells)')
plot_embedding(emb_m, mac_labels,   axes[1], f'Macaque (n={len(mac_cells)} cells)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'A2_cebra_embeddings.pdf'))
plt.show()

In [ ]:
# Within-species: silhouette of load clusters in latent space
def load_silhouette(emb, labels):
    if emb is None: return np.nan
    return silhouette_score(emb, labels['load'].to_numpy())

sil_h = load_silhouette(emb_h, human_labels)
sil_m = load_silhouette(emb_m, mac_labels)
print(f'Load-cluster silhouette: human={sil_h:.3f}, macaque={sil_m:.3f}')
print('(Higher = loads better separated in the manifold. Range: -1 to 1.)')

In [ ]:
# Cross-species: Procrustes alignment of per-load centroids
from scipy.spatial import procrustes

def per_load_centroids(emb, labels):
    cents = np.array([emb[labels['load'] == L].mean(axis=0) for L in [1, 2, 3]])
    return cents

if emb_h is not None and emb_m is not None:
    cent_h = per_load_centroids(emb_h, human_labels)
    cent_m = per_load_centroids(emb_m, mac_labels)
    # Procrustes returns standardized matrices and disparity (residual alignment error)
    mtx1, mtx2, disparity = procrustes(cent_h, cent_m)
    print(f'Procrustes disparity (human <-> macaque load centroids): {disparity:.4f}')
    print('(Lower = better alignment. 0 = identical structure, 1 = orthogonal.)')

    # Permutation null: shuffle macaque load labels, recompute disparity
    null_disp = []
    n_perm = 1000
    rng = np.random.default_rng(RNG_SEED)
    for _ in range(n_perm):
        shuf = mac_labels.copy()
        shuf['load'] = rng.permutation(shuf['load'].to_numpy())
        cent_m_shuf = per_load_centroids(emb_m, shuf)
        try:
            _, _, d = procrustes(cent_h, cent_m_shuf)
            null_disp.append(d)
        except Exception:
            null_disp.append(np.nan)
    null_disp = np.array(null_disp)
    p_emp = (null_disp <= disparity).mean()
    print(f'Permutation p-value (disparity <= observed under null): {p_emp:.3f}')
    print(f'Null distribution: median={np.nanmedian(null_disp):.4f}, '
          f'5th pct={np.nanpercentile(null_disp, 5):.4f}')

In [ ]:
# Cross-species decoding: train load decoder on macaque embedding, test on human
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

if emb_h is not None and emb_m is not None:
    # First: within-species decoding (sanity check)
    clf_m = LogisticRegression(max_iter=2000, multi_class='multinomial')
    cv_m = cross_val_score(clf_m, emb_m, mac_labels['load'].to_numpy(), cv=5).mean()
    clf_h = LogisticRegression(max_iter=2000, multi_class='multinomial')
    cv_h = cross_val_score(clf_h, emb_h, human_labels['load'].to_numpy(), cv=5).mean()
    print(f'Within-species load decoding (5-fold CV):')
    print(f'  Macaque: {cv_m:.3f} (chance = 0.333)')
    print(f'  Human:   {cv_h:.3f} (chance = 0.333)')

    # Cross-species: align macaque to human latent space via Procrustes, then train+test
    # For Procrustes-aligned cross-decoding, we need same-dim spaces, which CEBRA gives us.
    # Train on macaque (post-alignment), test on human.
    from scipy.linalg import orthogonal_procrustes
    # Compute optimal rotation from mac centroids to human centroids
    R, _ = orthogonal_procrustes(cent_m, cent_h)
    emb_m_aligned = emb_m @ R
    clf_x = LogisticRegression(max_iter=2000, multi_class='multinomial')
    clf_x.fit(emb_m_aligned, mac_labels['load'].to_numpy())
    cross_acc = clf_x.score(emb_h, human_labels['load'].to_numpy())
    print(f'\nCross-species load decoding (train macaque, test human, after Procrustes):')
    print(f'  Accuracy: {cross_acc:.3f} (chance = 0.333)')

    # Permutation null for cross-decoding
    rng = np.random.default_rng(RNG_SEED)
    null_acc = []
    for _ in range(500):
        shuf_labels = rng.permutation(human_labels['load'].to_numpy())
        null_acc.append(clf_x.score(emb_h, shuf_labels))
    null_acc = np.array(null_acc)
    p_cross = (null_acc >= cross_acc).mean()
    print(f'  Permutation p (null = shuffled human labels): {p_cross:.3f}')
    print(f'  Null: median={np.median(null_acc):.3f}, 95th pct={np.percentile(null_acc, 95):.3f}')

## Cell 5 — Interpretation guide

Read the outputs above as follows.

### Load-firing analysis (Cell 3)
- **β_load > 0 in both species**: firing increases with load. Consistent with persistent-activity load tracking.
- **β_load < 0 in both species**: firing decreases with load. The Kaminski 2017 finding for pre-SMA. Consistent with a generic effort/normalization signal, not specific to gain.
- **Opposite signs**: species differ in basic load-rate relationship; cross-species similarity claim weakens.
- **Human load × present interaction significant + negative**: present-vs-absent contrast attenuates with load. This is the gain-like signature without claiming gain mechanism.

### CEBRA analysis (Cell 4)
- **Within-species silhouette ≫ 0 in both species**: load is well-organized in the manifold. Human silhouette ~0 with macaque ≫ 0 means the human side is too noisy/small for manifold conclusions.
- **Procrustes disparity below null 5th percentile**: load-centroid geometry is more similar across species than chance. Supports manifold-level convergence.
- **Cross-decoding accuracy > chance with permutation p < 0.05**: load is decodable across species after alignment. Strongest evidence for shared manifold structure with respect to load.
- **All cross-species statistics at chance**: no detectable manifold-level convergence. This isn't necessarily evidence of *different* architectures — could just be insufficient power on the human side.

### What you can claim from positive results
- *"Human pre-SMA/dACC concept cells and macaque DMFC identity-tuned cells show convergent load modulation of delay-period firing."*
- *"The population manifold organized with respect to load shows above-chance alignment across species, suggesting that working-memory maintenance representations have shared geometric properties despite task structural differences."*

### What you cannot claim
- That humans and macaques share gain coding architecture. CEBRA + load-firing don't test this.
- That the alignment is mechanistic. Convergent task demands can produce convergent manifolds.
- Strong negative results on the human side. n=17 is underpowered for null claims.

### Robustness checks worth running
1. **Re-run CEBRA with different `output_dimension`** (3, 4, 8). If alignment is real, it should be stable across reasonable choices.
2. **Re-run with shuffled-label CEBRA training** as an additional null. Trained-on-shuffles models should not show cross-species alignment.
3. **Bootstrap the macaque side at session level** (resample sessions, then cells within session) and re-fit CEBRA. If the result is driven by a few sessions, alignment will be unstable.
4. **Vary the human cell-selection threshold.** Add a `concept_score > 0.5` filter and see if the analysis is sensitive to which cells you call concept cells.
5. **Run on incorrect trials only** (if enough exist). Errors should *not* show clean load structure if the manifold is doing memory work.